# ASL Alphabet Classification Model Training

This notebook handles data loading, preprocessing, and model training

## IMPORT & SETUP

In [23]:
import json
from tensorflow.keras import layers
import tensorflow as tf
import os
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Create training and model output directory if it doesn't exist
os.makedirs('../model_training', exist_ok=True)
os.makedirs('../model_outputs', exist_ok=True)

## LOAD DATASET

In [7]:
df = pd.read_csv('../dataset/landmark_dataset.csv')
print(df['label'].value_counts())

X = df.drop('label', axis=1).values  # 63 features
y = df['label'].values

label
A        200
B        200
C        200
D        200
E        200
F        200
G        200
H        200
I        200
K        200
L        200
M        200
N        200
O        200
P        200
Q        200
R        200
S        200
T        200
U        200
V        200
W        200
X        200
Y        200
del      200
space    200
Name: count, dtype: int64


## ENCODE LABELS

In [13]:
encoder = LabelEncoder()
y_encoded = encoder.fit_transform(y)
class_names = list(encoder.classes_)
print("Classes:", class_names)

Classes: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'del', 'space']


## DATA SPLITTING (80/10/10)

In [14]:
X_train, X_temp, y_train, y_temp = train_test_split(X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")

Train: 4160 | Val: 520 | Test: 520


## MODEL ARCHITECTURE

In [15]:
model = tf.keras.Sequential([
    layers.Input(shape=(63,)),
    layers.Dense(128, activation='relu', kernel_initializer='he_normal'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(64, activation='relu', kernel_initializer='he_normal'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(len(class_names), activation='softmax'),
])

model.summary(print_fn=lambda x: print(x, flush=True))

Model: "sequential_3"
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_9 (Dense)                 │ (None, 128)            │         8,192 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_6           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_7           │ (None, 64)        

### COMPILE MODEL

In [16]:
optimizer=tf.keras.optimizers.Adam(learning_rate=0.001)

model.compile(
    optimizer=optimizer,
    loss= tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    metrics=['accuracy'],
)

### CALLBACKS SETUP

In [17]:
callbacks = [

    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=7,
        restore_best_weights=True
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1
    ),

    tf.keras.callbacks.ModelCheckpoint(
        filepath='../model_outputs/best_model.keras',
        monitor='val_accuracy',
        save_best_only=True
    )
]

### MODEL TRAINING

In [18]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=32,
    callbacks=callbacks
)

130/130 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9993 - loss: 0.0079 - val_accuracy: 1.0000 - val_loss: 1.5225e-04 - learning_rate: 1.9531e-06


In [19]:
history.history

{'accuracy': [0.5060096383094788,
  0.8939903974533081,
  0.9569711685180664,
  0.9771634340286255,
  0.9846153855323792,
  0.9887019395828247,
  0.9894230961799622,
  0.9932692050933838,
  0.9937499761581421,
  0.9906250238418579,
  0.9925480484962463,
  0.9935095906257629,
  0.9947115182876587,
  0.9944711327552795,
  0.9949519038200378,
  0.9966346025466919,
  0.996874988079071,
  0.998317301273346,
  0.9966346025466919,
  0.9980769157409668,
  0.9975961446762085,
  0.998317301273346,
  0.9966346025466919,
  0.998317301273346,
  0.9975961446762085,
  0.9975961446762085,
  0.998317301273346,
  0.9985576868057251,
  0.998317301273346,
  0.9992788434028625,
  0.9992788434028625,
  0.9990384578704834,
  0.9975961446762085,
  0.9985576868057251,
  0.9992788434028625,
  0.9987980723381042,
  0.9985576868057251,
  0.9990384578704834,
  0.9990384578704834,
  0.9990384578704834,
  0.9985576868057251,
  0.9978365302085876,
  0.9985576868057251,
  0.9990384578704834,
  0.9990384578704834,
  0.

In [20]:
model.evaluate(X_test, y_test)

[0.00022892608831170946, 1.0]

### EXPORT ARTIFACTS

In [25]:
# Saves and Export Datasets
with open('../model_outputs/class_names.json', 'w') as f:
    json.dump(class_names, f)